# Bi-LSTM Training (AIVIEW)

Trains two **separate** bi-LSTM models on the extracted First Impressions features, one per modality:

- **Audio-only model**: sequence of 15 VGGish embeddings (128-dim each) → input `(batch, 15, 128)`
- **Visual-only model**: sequence of 30 VGG-Face frame embeddings (4096-dim each) → input `(batch, 30, 4096)`

Each model independently predicts the 5 OCEAN personality traits in `[0, 1]` (sigmoid + MSE).

Runs on GPU (NVIDIA RTX 4050 Laptop, 6 GB VRAM) with PyTorch, mixed precision (AMP) for higher GPU throughput.

In [ ]:
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=np.exceptions.VisibleDeprecationWarning)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__)
print('device:', device)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| VRAM {p.total_memory / 1e9:.1f} GB')

In [ ]:
# ----------------------------------------------------------------------------
# Hyperparameters & paths
# ----------------------------------------------------------------------------
MODALITIES = {
    'audio': {
        'feature_dir': 'datasets/extracted/Features/audio',
        'seq_len': 15,
        'feat_dim': 128,
        'hidden': 64,
        'num_layers': 1,
        'dropout': 0.3,
        'batch_size': 32,
        'lr': 1e-4,
        'weight_decay': 1e-5,
        'epochs': 100,
        'patience': 10,
    },
    'visual': {
        'feature_dir': 'datasets/extracted/Features/visual',
        'seq_len': 30,
        'feat_dim': 4096,
        'hidden': 64,
        'num_layers': 1,
        'dropout': 0.3,
        'batch_size': 32,
        'lr': 1e-4,
        'weight_decay': 1e-5,
        'epochs': 100,
        'patience': 10,
    },
}

LABEL_PKL = {
    'trainingData': 'code_reference/First-Impression/Annotations/annotation_training.pkl',
    'validationData': 'code_reference/First-Impression/Annotations/annotation_validation.pkl',
}

OCEAN = ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

## 1. Load OCEAN labels

The annotation files map `<clip>.mp4` → score in `[0, 1]` for each of the 6 traits. We keep the 5 OCEAN traits (drop `interview`) as the regression targets.

In [ ]:
def load_labels(split):
    with open(LABEL_PKL[split], 'rb') as f:
        ann = pickle.load(f, encoding='latin1')
    mapping = {}
    for trait in OCEAN:
        for name, score in ann[trait].items():
            mapping.setdefault(name.replace('.mp4', ''), {})[trait] = float(score)
    return {name: np.array([m[t] for t in OCEAN], dtype=np.float32)
            for name, m in mapping.items()}

LABELS_TRAIN = load_labels('trainingData')
LABELS_VAL = load_labels('validationData')
print('clips with labels -> train:', len(LABELS_TRAIN), '| val:', len(LABELS_VAL))

## 2. Load & normalize features

Reads `feature{0..N}.npy` for every clip and stacks them into `(n_clips, seq_len, feat_dim)`.
Incomplete clips (wrong number of frames/segments) are skipped.
Features are z-score normalized **per feature-dimension using training-set statistics only**.

Data is moved to the GPU once so all batching happens on-device (visual train+val ≈ 0.8 GB < 6 GB VRAM).

In [ ]:
def load_features(split, mod):
    cfg = MODALITIES[mod]
    root = Path(cfg['feature_dir']) / split
    clips, arrays = [], []
    dirs = sorted(root.iterdir())
    for clip_dir in tqdm(dirs, desc=f'load {mod}/{split}', leave=False):
        if not clip_dir.is_dir():
            continue
        try:
            stack = [np.load(clip_dir / f'feature{i}.npy', allow_pickle=False)
                     for i in range(cfg['seq_len'])]
        except (FileNotFoundError, OSError):
            continue
        arr = np.stack(stack, axis=0).astype(np.float32)
        if arr.shape != (cfg['seq_len'], cfg['feat_dim']):
            continue
        arrays.append(arr)
        clips.append(clip_dir.name)
    return clips, np.stack(arrays, axis=0)


def build_dataset(mod, verbose=True):
    cfg = MODALITIES[mod]

    tr_clips, tr_X = load_features('trainingData', mod)
    va_clips, va_X = load_features('validationData', mod)

    tr_keep = [i for i, c in enumerate(tr_clips) if c in LABELS_TRAIN]
    va_keep = [i for i, c in enumerate(va_clips) if c in LABELS_VAL]
    tr_clips = [tr_clips[i] for i in tr_keep]
    tr_X = tr_X[tr_keep]
    va_clips = [va_clips[i] for i in va_keep]
    va_X = va_X[va_keep]

    mean = tr_X.mean(axis=(0, 1), keepdims=True)
    std = tr_X.std(axis=(0, 1), keepdims=True) + 1e-8
    tr_X = (tr_X - mean) / std
    va_X = (va_X - mean) / std

    tr_y = np.stack([LABELS_TRAIN[c] for c in tr_clips], axis=0)
    va_y = np.stack([LABELS_VAL[c] for c in va_clips], axis=0)

    if verbose:
        print(f'[{mod}] train: {tr_X.shape} -> labels {tr_y.shape} | '
              f'val: {va_X.shape} -> labels {va_y.shape}')

    tr = (torch.from_numpy(tr_X).to(device), torch.from_numpy(tr_y).to(device))
    va = (torch.from_numpy(va_X).to(device), torch.from_numpy(va_y).to(device))
    return tr, va

## 3. Model definition

`AiviewBiLSTM`: bidirectional LSTM over the feature sequence; the final **forward + backward**
hidden states are concatenated and fed to small MLP head producing 5 sigmoid outputs (one per OCEAN trait).

In [ ]:
class AiviewBiLSTM(nn.Module):
    def __init__(self, feat_dim, hidden, num_layers, dropout, out_dim=5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=hidden,
                            num_layers=num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)                  # h_n: (2*num_layers, B, hidden)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)    # forward + backward last states
        return self.head(h)

## 4. Training / evaluation utilities

- Loss: MSE on sigmoid outputs; metric: MAE.
- AMP (`autocast` + `GradScaler`) so the 4096-dim visual stream trains fast on the 6 GB GPU.
- Early stopping on **validation MAE** with best-state restore.

Models are saved to `models/bilstm_<modality>.pt` along with a JSON history.

In [ ]:
def train_one_epoch(model, X, y, optimizer, scaler, batch_size, loss_fn):
    model.train()
    total_loss, total_mae, n = 0.0, 0.0, 0
    perm = torch.randperm(X.size(0), device=X.device)
    for i in range(0, X.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X[idx], y[idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=scaler.is_enabled()):
            pred = model(xb)
            loss = loss_fn(pred, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * xb.size(0)
        total_mae += (pred - yb).abs().sum().item()
        n += xb.size(0)
    return total_loss / n, total_mae / n


@torch.no_grad()
def evaluate(model, X, y, batch_size, loss_fn):
    model.eval()
    total_loss, per_trait, n = 0.0, [], 0
    for i in range(0, X.size(0), batch_size):
        xb, yb = X[i:i + batch_size], y[i:i + batch_size]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=torch.cuda.is_available()):
            pred = model(xb)
            loss = loss_fn(pred, yb)
        total_loss += loss.item() * xb.size(0)
        per_trait.append((pred - yb).abs().mean(dim=0).cpu().numpy())
        n += xb.size(0)
    per_trait = np.array(per_trait).mean(axis=0)
    return total_loss / n, per_trait.mean(), per_trait


def train_model(mod):
    cfg = MODALITIES[mod]
    (Xtr, ytr), (Xva, yva) = build_dataset(mod)

    model = AiviewBiLSTM(cfg['feat_dim'], cfg['hidden'], cfg['num_layers'], cfg['dropout']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    loss_fn = nn.MSELoss()
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    best_mae, best_state, best_epoch, patience_left = float('inf'), None, 0, cfg['patience']
    history = {'epoch': [], 'train_loss': [], 'train_mae': [], 'val_loss': [], 'val_mae': []}

    for epoch in range(1, cfg['epochs'] + 1):
        tl, tmae = train_one_epoch(model, Xtr, ytr, optimizer, scaler, cfg['batch_size'], loss_fn)
        vl, vmae, _ = evaluate(model, Xva, yva, cfg['batch_size'], loss_fn)
        history['epoch'].append(epoch)
        history['train_loss'].append(float(tl))
        history['train_mae'].append(float(tmae))
        history['val_loss'].append(float(vl))
        history['val_mae'].append(float(vmae))

        if vmae < best_mae:
            best_mae = vmae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_left = cfg['patience']
        else:
            patience_left -= 1

        print(f'[{mod}] epoch {epoch:3d}/{cfg["epochs"]} | train loss {tl:.4f} mae {tmae:.4f} | '
              f'val loss {vl:.4f} mae {vmae:.4f} | best {best_mae:.4f} (ep{best_epoch})')

        if patience_left <= 0:
            print(f'[{mod}] early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    model.eval()
    _, best_mae, mae_per_trait = evaluate(model, Xva, yva, cfg['batch_size'], loss_fn)

    ckpt = {'modality': mod, 'state_dict': best_state, 'traits': OCEAN, 'config': cfg}
    torch.save(ckpt, MODEL_DIR / f'bilstm_{mod}.pt')

    meta = {'modality': mod, 'best_epoch': best_epoch, 'best_va_mae': float(best_mae),
            'val_mae_per_trait': mae_per_trait.tolist(), 'traits': OCEAN,
            'config': cfg, 'history': history}
    with open(MODEL_DIR / f'bilstm_{mod}_history.json', 'w') as f:
        json.dump(meta, f, indent=2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='train')
    axes[0].plot(history['epoch'], history['val_loss'], label='val')
    axes[0].set_title(f'{mod} loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(history['epoch'], history['train_mae'], label='train')
    axes[1].plot(history['epoch'], history['val_mae'], label='val')
    axes[1].set_title(f'{mod} MAE'); axes[1].set_xlabel('epoch'); axes[1].legend()
    plt.tight_layout(); plt.show()

    print(f'\n[{mod}] BEST val MAE = {best_mae:.4f} at epoch {best_epoch}')
    for t, m in zip(OCEAN, mae_per_trait):
        print(f'    {t:16s} MAE = {m:.4f}')
    print(f'    saved: {MODEL_DIR / f"bilstm_{mod}.pt"}')
    return meta

## 5. Train audio-only bi-LSTM

Input: the 15 VGGish segment embeddings per clip. **Audio training runs independently** — it does not use any visual data.

In [ ]:
audio_result = train_model('audio')

## 6. Train visual-only bi-LSTM

Input: the 30 VGG-Face frame embeddings per clip. **Visual training runs separately** from audio.

In [ ]:
visual_result = train_model('visual')

## 7. Result summary

Final validation MAE (overall + per OCEAN trait) for both modalities, plus where each model is saved.

In [ ]:
results = []
for mod in MODALITIES:
    with open(MODEL_DIR / f'bilstm_{mod}_history.json') as f:
        results.append(json.load(f))

print(f"{'modality':<8} | {'best val MAE':<13} | " + ' | '.join(f'{t[:4]}' for t in OCEAN))
print('-' * (8 + 15 + 5 * len(OCEAN)))
for r in results:
    per = r['val_mae_per_trait']
    print(f"{r['modality']:<8} | {r['best_va_mae']:<13.4f} | " + ' | '.join(f'{m:.3f}' for m in per))
    print(f"    best epoch: {r['best_epoch']}  ->  models/bilstm_{r['modality']}.pt")